In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 假设你的项目结构已经配好，可以直接导入
from core.vol_surface import VolSurface
from core.dupire import LocalVolPricer

# ---------------------------------------------------------
# 1. 准备数据：伪造一个极其真实的“波动率微笑”市场数据
# ---------------------------------------------------------
S0 = 100.0
r = 0.05
q = 0.02

# 设定行权价和到期时间的网格
# K 从 70 到 130，T 从 0.1年 到 2.0年
K_grid = np.linspace(70, 130, 30)
T_grid = np.linspace(0.1, 2.0, 30)

# 生成一个带有 Smile (K方向) 和 Term Structure (T方向) 的 IV 矩阵
# 几何意义：随着 K 偏离平值 100，波动率上升；随着 T 变长，波动率趋于平缓
iv_matrix = np.zeros((len(K_grid), len(T_grid)))
for i, K in enumerate(K_grid):
    for j, T in enumerate(T_grid):
        # 这是一个经验公式，用来模拟真实的期权面貌
        smile = 0.15 + 0.0001 * (K - 100)**2 
        term_structure = 0.05 / np.sqrt(T)
        iv_matrix[i, j] = smile + term_structure

# 实例化我们的模型
vol_surface = VolSurface(K_grid, T_grid, iv_matrix)
lv_pricer = LocalVolPricer(vol_surface, S0, r, q)

# ---------------------------------------------------------
# 2. 计算密集的 Local Volatility 网格用于绘图
# ---------------------------------------------------------
# 为了画图平滑，我们生成更细的网格 (Meshgrid)
K_plot, T_plot = np.meshgrid(np.linspace(80, 120, 40), np.linspace(0.2, 1.5, 40))
IV_plot = np.zeros_like(K_plot)
LV_plot = np.zeros_like(K_plot)

print("正在努力用 Dupire 公式逐点计算局部波动率，请稍候...")
for i in range(K_plot.shape[0]):
    for j in range(K_plot.shape[1]):
        k = K_plot[i, j]
        t = T_plot[i, j]
        # 提取平滑后的 IV
        IV_plot[i, j] = vol_surface.get_iv(k, t)
        # 用有限差分计算 LV
        LV_plot[i, j] = lv_pricer.local_vol(k, t)

# ---------------------------------------------------------
# 3. 3D 可视化渲染 (魔法时刻！)
# ---------------------------------------------------------
fig = plt.figure(figsize=(16, 7))

# --- 左图：Implied Volatility (隐含波动率曲面) ---
ax1 = fig.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(K_plot, T_plot, IV_plot, cmap='viridis', edgecolor='none', alpha=0.9)
ax1.set_title('Implied Volatility Surface (The "Average")', fontsize=14, pad=15)
ax1.set_xlabel('Strike (K)', fontsize=12)
ax1.set_ylabel('Time to Maturity (T)', fontsize=12)
ax1.set_zlabel('Volatility', fontsize=12)
ax1.view_init(elev=25, azim=-120)  # 调整观测视角
fig.colorbar(surf1, ax1=ax1, shrink=0.5, aspect=10)

# --- 右图：Local Volatility (局部波动率曲面) ---
ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(K_plot, T_plot, LV_plot, cmap='plasma', edgecolor='none', alpha=0.9)
ax2.set_title('Local Volatility Surface (The "Instantaneous")', fontsize=14, pad=15)
ax2.set_xlabel('Strike (K)', fontsize=12)
ax2.set_ylabel('Time to Maturity (T)', fontsize=12)
ax2.set_zlabel('Volatility', fontsize=12)
ax2.view_init(elev=25, azim=-120)
fig.colorbar(surf2, ax2=ax2, shrink=0.5, aspect=10)

plt.tight_layout()
plt.show()